# GPU Released Memory Failure — AhmetFurkanDEMIR/Generative-Adversarial-Networks-GAN

**Smell (`GANs.py` lines 16, 36, 83):** Generator (`g`), discriminator (`d`), and DCGAN (`dcgan`) are built inside `train()` with no `K.clear_session()` call between repeated invocations. GPU memory for all three models accumulates across 10 runs.

**Fix:** Call `K.clear_session()` + `del` models + `gc.collect()` after each run.

In [ ]:
!pip install -q codecarbon pillow

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, BatchNormalization, Activation,
                                      Reshape, UpSampling2D, Conv2D, ELU,
                                      Flatten, Dropout)
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.datasets import mnist
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS      = 10
NUM_EPOCHS  = 2    # reduced from 5 for Kaggle feasibility
BATCH_SIZE  = 32

(x_train, _), (_, _) = mnist.load_data()
x_train = (x_train.astype(np.float32) - 127.5) / 127.5
x_train = x_train.reshape(x_train.shape[0], x_train.shape[1], x_train.shape[2], 1)
print('MNIST loaded:', x_train.shape)

In [ ]:
# ── Model builders from AhmetFurkanDEMIR/Generative-Adversarial-Networks-GAN/GANs.py ──
def build_generator(input_dim=100, units=1024, activation='relu'):
    model = Sequential([
        Dense(input_dim=input_dim, units=units),
        BatchNormalization(), Activation(activation),
        Dense(128*7*7), BatchNormalization(), Activation(activation),
        Reshape((7,7,128)),
        UpSampling2D((2,2)), Conv2D(64,(5,5),padding='same'),
        BatchNormalization(), Activation(activation),
        UpSampling2D((2,2)), Conv2D(1,(5,5),padding='same'),
        Activation('tanh'),
    ])
    return model

def build_discriminator(input_shape=(28,28,1), nb_filter=64):
    model = Sequential([
        Conv2D(nb_filter,(5,5),strides=(2,2),padding='same',input_shape=input_shape),
        BatchNormalization(), ELU(),
        Conv2D(2*nb_filter,(5,5),strides=(2,2)),
        BatchNormalization(), ELU(),
        Flatten(), Dense(4*nb_filter), BatchNormalization(), ELU(),
        Dropout(0.5), Dense(1), Activation('sigmoid'),
    ])
    return model

def run_train(x_data, num_epochs=NUM_EPOCHS):
    lr = 0.0002
    optimize = Adam(learning_rate=lr, beta_1=0.5)
    g = build_generator()
    d = build_discriminator()
    d.trainable = True
    d.compile(loss='binary_crossentropy', metrics=['accuracy'], optimizer=optimize)
    d.trainable = False
    dcgan = Sequential([g, d])
    dcgan.compile(loss='binary_crossentropy', metrics=['accuracy'], optimizer=optimize)

    num_batches = x_data.shape[0] // BATCH_SIZE
    y_d_true = np.ones(BATCH_SIZE)
    y_d_gen  = np.zeros(BATCH_SIZE)
    y_g      = np.ones(BATCH_SIZE)

    for epoch in range(num_epochs):
        for i in range(min(num_batches, 200)):  # cap batches per epoch
            x_d_batch = x_data[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
            x_g = np.random.normal(0, 0.5, (BATCH_SIZE, 100))
            x_d_gen = g.predict(x_g, verbose=0)
            d.trainable = True
            d.train_on_batch(x_d_batch, y_d_true)
            d.train_on_batch(x_d_gen,   y_d_gen)
            d.trainable = False
            dcgan.train_on_batch(x_g, y_g)
        print(f'  epoch {epoch+1}/{num_epochs} done')
    return g, d, dcgan

print('Model builders ready')

## BEFORE — Smell Active (no cleanup between runs)

In [ ]:
results_before = []

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    # ❌ SMELL: no K.clear_session() — g, d, dcgan from previous run stay in GPU memory
    tracker = EmissionsTracker(
        project_name=f'AhmetGAN_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    g, d, dcgan = run_train(x_train)
    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run,
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    # ❌ SMELL: no cleanup

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/AhmetFurkanDEMIR_GAN_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/AhmetFurkanDEMIR_GAN_before.csv')

## AFTER — Smell Fixed (K.clear_session + del + gc.collect between runs)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    # ✅ FIX: clear session before building new models
    K.clear_session()
    gc.collect()
    tracker = EmissionsTracker(
        project_name=f'AhmetGAN_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    g, d, dcgan = run_train(x_train)
    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run,
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    # ✅ FIX: explicit teardown
    del g, d, dcgan
    K.clear_session()
    gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/AhmetFurkanDEMIR_GAN_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/AhmetFurkanDEMIR_GAN_after.csv')

In [ ]:
print('\n=== SUMMARY: AhmetFurkanDEMIR/Generative-Adversarial-Networks-GAN ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")